# مترجم خودکار مانگا / مانهوا به فارسی

OCR → پاک‌سازی حباب → ترجمه با Gemini → رندر فارسی

**ترتیب:** سلول‌ها را از بالا به پایین با Shift+Enter اجرا کنید.

قبل از شروع: `Runtime → Change runtime type → GPU (T4)`

## ۱) نصب وابستگی‌ها

In [ ]:
!pip install -q numpy==1.26.4
!pip install -q --no-deps PyMuPDF opencv-python-headless Pillow google-genai \
    arabic-reshaper python-bidi requests beautifulsoup4
!pip install -q --no-deps imgaug scipy imageio matplotlib networkx shapely
!pip install -q --no-deps paddlepaddle==2.6.2 lmdb paddleocr==2.7.0.3
!pip install -q astor scikit-image pyclipper tqdm rapidfuzz

## ۲) فونت فارسی (Vazirmatn)

In [ ]:
!mkdir -p fonts
!wget -q -O fonts/Vazirmatn-Bold.ttf \
  https://github.com/rastikerdar/vazirmatn/raw/master/fonts/ttf/Vazirmatn-Bold.ttf
!ls -la fonts/

## ۳) اسکریپت مترجم
فایل `manga_translator.py` را از ریپو آپلود کنید، یا محتوای آن را در یک سلول `%%writefile` بنویسید.

In [ ]:
# اگر فایل را از GitHub کلون کرده‌اید:
# !git clone https://github.com/YOUR_USER/manga-translator-fa.git
# !cp manga-translator-fa/manga_translator.py .

from google.colab import files
print('فایل manga_translator.py را آپلود کنید (یا از سلول بالا کلون کنید)')
# uploaded = files.upload()  # در صورت نیاز فعال کنید

## ۴) ورودی
پوشه/ZIP/PDF تصاویر مانگا را آپلود کنید یا مسیر Google Drive بدهید.

In [ ]:
from google.colab import files
import os, zipfile

input_path = 'input_pages'  # پوشه یا فایل
os.makedirs(input_path, exist_ok=True)

print('فایل‌های تصویر / ZIP را آپلود کنید...')
uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as zf:
            zf.extractall(input_path)
        print(f'استخراج شد: {name}')
    else:
        os.rename(name, os.path.join(input_path, name))
        print(f'ذخیره شد: {name}')

print('محتوای ورودی:', os.listdir(input_path)[:20])

## ۵) اجرا
کلید Gemini را از [Google AI Studio](https://aistudio.google.com/apikey) بگیرید.

In [ ]:
import os

# یک یا چند کلید (با کاما)
os.environ['GEMINI_API_KEY'] = 'YOUR_KEY_1,YOUR_KEY_2'

input_path = 'input_pages'
output_path = 'output_pages_fa.pdf'

!python manga_translator.py \
  -i "{input_path}" \
  -o "{output_path}" \
  --font fonts/Vazirmatn-Bold.ttf \
  --ocr-lang en \
  --reading-order rtl \
  --max-width 900

# اگر قطع شد، دوباره همین سلول را اجرا کنید؛ صفحات قبلی از کش خوانده می‌شوند.

## ۶) دانلود خروجی

In [ ]:
from google.colab import files
files.download(output_path)